# Iterative DPO on Inkling (2 rounds, reasoning effort 0.5)

Runs 2 rounds of iterative DPO on `thinkingmachines/Inkling` over the
`nl_gameable` environment, driving the three pipeline stages directly with
`GenerateConfig` / `SelectConfig` / `TrainConfig` — the
`rewardhacking_training/iterative_training` step machine is **not** used.
Epoch indexing is done manually: the epoch-0 prompt order is shuffled once,
iteration 0 generates on the first half, iteration 1 on the second half, so
the two rounds together cover exactly one `nl_gameable` epoch.

Per iteration:

1. **Generate + score** — `run_generate` samples `N_SAMPLES=50` completions per
   prompt from the current model (base Inkling on round 0, the round-0 sampler
   weights on round 1); the programmatic graders score inside the inspect task.
2. **Select** — `run_select` builds DPO pairs from the eval log.
3. **Train** — `run_train` runs one tinker DPO pass, resuming from the previous
   round's optimizer state on round 1.


In [ ]:
import json
import math
import os
import random
import sys
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv

# run_select and tinker_cookbook's DPO trainer call asyncio.run(), which needs
# a nested loop inside Jupyter.
nest_asyncio.apply()

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "rewardhacking_training").exists():
    assert REPO_ROOT != REPO_ROOT.parent, "run from inside the repo"
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv()
assert os.environ.get("TINKER_API_KEY"), "TINKER_API_KEY not set"

from rewardhacking_training.generate.generate import (
    GenerateConfig,
    ModelConfig,
    run_generate,
)
from rewardhacking_training.generate.inference_client import InferenceClientConfig
from rewardhacking_training.select.select import SelectConfig, run_select
from rewardhacking_training.train.train import TrainConfig, run_train

MODEL = "thinkingmachines/Inkling"
RENDERER = "tml_v0"
EFFORT = 0.5
N_SAMPLES = 16          # completions per prompt (inspect epochs)
N_ITERATIONS = 2
SEED = 42
MAX_TOKENS = 1536      # renderer's token-budget heuristic for effort 0.5
MAX_CONNECTIONS = 200

RUN_NAME = "inkling-itdpo-effort05"
RUN_DIR = REPO_ROOT / "output" / "notebook_iterative_dpo" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"run dir: {RUN_DIR}")


## Reasoning effort 0.5

The tinker sampling path (`InspectAPIFromTinkerSampling.generate`) calls
`renderer.build_generation_prompt(convo)` with no `effort` argument, so Inkling
would always sample at the renderer default (0.9). Patch the default so every
generation in this process uses `EFFORT`. Run this cell before any generation.


In [ ]:
from tinker_cookbook.renderers import tml_v0

if not getattr(tml_v0.TmlV0Renderer.build_generation_prompt, "_effort_patched", False):
    _orig_build = tml_v0.TmlV0Renderer.build_generation_prompt

    def _build_with_effort(self, messages, role="assistant", prefill=None, effort=EFFORT):
        return _orig_build(self, messages, role=role, prefill=prefill, effort=effort)

    _build_with_effort._effort_patched = True
    tml_v0.TmlV0Renderer.build_generation_prompt = _build_with_effort

print(f"TmlV0Renderer default reasoning effort -> {EFFORT}")


## Manual epoch indexing

Load the `nl_gameable` task once to enumerate its prompt ids, shuffle the
epoch-0 order with a fixed seed, and split it in half: iteration 0 gets the
first half, iteration 1 the second half.


In [ ]:
TASK = "rewardhacking_training.envs.nl_gameable.nl_gameable_env:nl_gameable"
SYSTEM_PROMPTS_PATH = "rewardhacking_training/prompts/system_prompts/thinking_variants.json"
BASE_TASK_ARGS = {
    "scorer_mode": "programmatic",  # deterministic per-prompt graders
    "persona_only": True,           # native reasoner: skip the <think>-tag prompt variants
    "max_tokens": MAX_TOKENS,
}

from rewardhacking_training.envs.nl_gameable.nl_gameable_env import nl_gameable

_task = nl_gameable(**BASE_TASK_ARGS)
all_prompt_ids = [str(s.id) for s in _task.dataset]
print(f"nl_gameable epoch size: {len(all_prompt_ids)}")

epoch0 = list(all_prompt_ids)
random.Random(f"{SEED}/nl_gameable/epoch/0").shuffle(epoch0)
half = math.ceil(len(epoch0) / 2)
ITER_PROMPT_IDS = [epoch0[:half], epoch0[half:]]
for i, ids in enumerate(ITER_PROMPT_IDS):
    print(f"iteration {i}: {len(ids)} prompts")


## Per-iteration config builders

Round 0 samples from base Inkling; round 1 samples from the `tinker://` sampler
weights that round 0's training produced (read from `iter_00/train_result.json`)
and resumes training from its optimizer state.


In [ ]:
def iter_dir(i: int) -> Path:
    d = RUN_DIR / f"iter_{i:02d}"
    d.mkdir(parents=True, exist_ok=True)
    return d


def prev_train_result(i: int) -> dict:
    return json.loads((iter_dir(i - 1) / "train_result.json").read_text())


def make_generate_config(i: int) -> GenerateConfig:
    # Base Inkling on round 0; a tinker:// path routes to the trained sampler
    # weights afterwards (the inference client keys on the model string).
    model = MODEL if i == 0 else prev_train_result(i)["model"]
    return GenerateConfig(
        task=TASK,
        model=model,
        model_config=ModelConfig(
            include_reasoning=True,
            inference_client=InferenceClientConfig(
                provider="tinker",
                base_model=MODEL,
                tinker_renderer_name=RENDERER,
            ),
        ),
        n_samples=N_SAMPLES,
        system_prompts_path=SYSTEM_PROMPTS_PATH,
        task_args={**BASE_TASK_ARGS, "prompt_ids": ITER_PROMPT_IDS[i]},
        max_connections=MAX_CONNECTIONS,
        retry_on_failure=False
    )


def make_select_config(i: int) -> SelectConfig:
    return SelectConfig(
        input_path=str(iter_dir(i) / "generate"),
        output_path=str(iter_dir(i) / "dpo_data.jsonl"),
        mode="dpo",
        seed=SEED,
    )


def make_train_config(i: int) -> TrainConfig:
    resume = None if i == 0 else prev_train_result(i)["resume_handle"]
    return TrainConfig(
        method="dpo",
        provider="tinker",
        base_model=MODEL,
        training_file=str(iter_dir(i) / "dpo_data.jsonl"),
        resume_handle=resume,
        suffix=f"{RUN_NAME}-it{i:02d}"[:40],
        wandb_name=f"{RUN_NAME}-it{i:02d}",
        output_dir=str(iter_dir(i)),
        tinker_renderer_name=RENDERER,
    )


def save_train_result(i: int, result) -> None:
    (iter_dir(i) / "train_result.json").write_text(
        json.dumps(
            {"model": result.model, "resume_handle": result.resume_handle, "info": result.info},
            indent=2,
        )
    )


## Iteration 0

### Generate + score

In [ ]:
gen_cfg = make_generate_config(0)
from dataclasses import asdict
for k, v in asdict(gen_cfg).items():
    print(k, v)

In [ ]:
gen_cfg = make_generate_config(0)
log_location = run_generate(gen_cfg, out=iter_dir(0) / "generate")
print(log_location)


### Select

In [ ]:
sel_cfg = make_select_config(0)
dpo_path = run_select(sel_cfg, out=iter_dir(0))
print(dpo_path)


### Train

In [ ]:
train_cfg = make_train_config(0)
train_result = run_train(train_cfg)
save_train_result(0, train_result)
print(train_result.model)


## Iteration 1

### Generate + score

In [ ]:
gen_cfg = make_generate_config(1)
log_location = run_generate(gen_cfg, out=iter_dir(1) / "generate")
print(log_location)


### Select

In [ ]:
sel_cfg = make_select_config(1)
dpo_path = run_select(sel_cfg, out=iter_dir(1))
print(dpo_path)


### Train

In [ ]:
train_cfg = make_train_config(1)
train_result = run_train(train_cfg)
save_train_result(1, train_result)
print(train_result.model)


## Final model

In [ ]:
final = prev_train_result(N_ITERATIONS)
(RUN_DIR / "final_model.txt").write_text(final["model"] + "\n")
print(f"final model: {final['model']}")
print(f"resume handle: {final['resume_handle']}")
